# PGAC Phase 2 v2 — Probe Training (lessons from first run applied)

**Target**: linear probe at residual L11 predicts top-k SAE features at L31 / L55.

**Lessons applied from first run (2026-05-02)**:
- Corpus auto-expanded to ~40K residuals (initial run hit 2243 from short prompts → undertrained)
- Eval batched (batch=256) — first run hung 19min on 2GB matmul
- Per-feature AUROC OPTIONAL (default off). Enable via `compute_per_feature_auroc=True`
- Resume-safe: re-running notebook loads cached residuals/features
- Hook path auto-resolution with sanity check

**Eval thresholds** (per `01_theoretical_bound.md` revised quality bound):
- 🟢 STRONG: recall@k ≥ 0.85 → 3-5x speedup at 8pp quality loss
- 🟡 PARTIAL: recall@k ∈ [0.5, 0.85] OR top-(2k) ≥ 0.85 → 2-3x speedup
- 🔴 INSUFFICIENT: recall@k < 0.5 AND recall@2k < 0.85 → architectural revision

**Compute**: ~3-4h Blackwell. **Cost**: ~$3.
**Drive**: `/content/drive/MyDrive/openinterp_runs/pgac_phase2/`


## Phase A — Setup + Drive


In [ ]:
from pathlib import Path
import os, json, time, gc
import torch, numpy as np
import torch.nn as nn, torch.nn.functional as F

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE = Path('/content/drive/MyDrive')
OUT = DRIVE / 'openinterp_runs' / 'pgac_phase2'
OUT.mkdir(parents=True, exist_ok=True)
print(f'OUT: {OUT}')
print(f'Existing: {sorted(p.name for p in OUT.iterdir())}')


In [ ]:
!pip install -q -U torchao
!pip install -q -U transformers accelerate datasets safetensors
!pip install -q -U huggingface_hub openinterp
!pip install -q scikit-learn matplotlib
import openinterp
print(f'✓ openinterp v{openinterp.__version__}')


## Phase B — Load Qwen3.6-27B + SAE encoders (HF)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login, hf_hub_download
from safetensors.torch import load_file
import getpass

CFG = {
    'model_id':                       'Qwen/Qwen3.6-27B',
    'sae_repo':                       'caiovicentino1/qwen36-27b-sae-papergrade',
    'sae_layers':                     [11, 31, 55],
    'd_model':                        5120,
    'd_sae':                          65536,
    'k_sae':                          128,            # verified from cfg.json
    'output_repo':                    'caiovicentino1/openinterp-pgac-phase2-probe',
    'random_seed':                    42,
    # Corpus settings
    'n_gsm8k':                        30,
    'n_simpleqa':                     20,
    'n_wiki':                         40,             # long sequences for diversity
    'max_seq_len':                    1024,
    # Training
    'n_epochs':                       5,
    'batch_size_train':               512,
    'lr':                             1e-3,
    'weight_decay':                   1e-5,
    # Eval
    'batch_size_eval':                256,            # batched to avoid 2GB matmul hang
    'compute_per_feature_auroc':      False,          # SLOW — enable only if needed
    'auroc_n_features':               200,
}
torch.manual_seed(CFG['random_seed']); np.random.seed(CFG['random_seed'])

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
login(HF_TOKEN, add_to_git_credential=False)

device = 'cuda'
tok = AutoTokenizer.from_pretrained(CFG['model_id'])
model = AutoModelForCausalLM.from_pretrained(CFG['model_id'], torch_dtype=torch.bfloat16, device_map='auto')
model.eval()
print(f'✓ Base model loaded: {torch.cuda.get_device_name(0)}')


In [ ]:
SAE_ENCODERS = {}
for L in CFG['sae_layers']:
    p = hf_hub_download(CFG['sae_repo'], f'sae_L{L}_latest.safetensors')
    state = load_file(p)
    SAE_ENCODERS[L] = {
        'W_enc': state['W_enc'].float(),
        'b_enc': state['b_enc'].float(),
    }
    print(f'  L{L} encoder: W_enc {tuple(SAE_ENCODERS[L]["W_enc"].shape)}')
print(f'✓ Loaded {len(SAE_ENCODERS)} SAE encoders')


## Phase C — Hooks + corpus + residual capture (resume-safe)

Auto-skips if `residuals_per_layer.pt` already on Drive.


In [ ]:
captured = {}

def make_capture_hook(layer_idx):
    def hook(module, input, output):
        h = output[0] if isinstance(output, tuple) else output
        captured[layer_idx] = h.detach().cpu()
    return hook

def get_layers_module(model):
    candidates = [
        ('model.layers',                lambda m: m.model.layers),
        ('model.language_model.layers', lambda m: m.model.language_model.layers),
    ]
    for name, getter in candidates:
        try:
            layers = getter(model)
            print(f'  ✓ Resolved layers at: {name} (n_layers={len(layers)})')
            return layers
        except AttributeError: continue
    raise RuntimeError('Could not locate transformer layers')

transformer_layers = get_layers_module(model)
all_layers = sorted(SAE_ENCODERS.keys())
hook_handles = []
for L in all_layers:
    h = transformer_layers[L].register_forward_hook(make_capture_hook(L))
    hook_handles.append(h)
print(f'✓ Hooks at L{all_layers}')

# Sanity check
test_input = tok('test', return_tensors='pt').to(device)
captured.clear()
with torch.no_grad():
    _ = model(**test_input)
for L in all_layers:
    if L not in captured:
        print(f'⚠️ L{L} hook did NOT fire')
    else:
        norm = captured[L].float().norm().item()
        print(f'  ✓ L{L} sanity fires: shape {tuple(captured[L].shape)}, norm {norm:.2f}')
captured.clear()


In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

residuals_path = OUT / 'residuals_per_layer.pt'
all_residuals = {L: [] for L in all_layers}

if residuals_path.exists():
    print(f'✓ Loading cached residuals from {residuals_path}')
    all_residuals = torch.load(residuals_path, weights_only=True)
    for L in all_layers:
        if L in all_residuals:
            print(f'  L{L}: {all_residuals[L].shape}')
else:
    print('Building corpus...')
    rng = np.random.default_rng(CFG['random_seed'])
    corpus = []
    # GSM8K
    try:
        gsm = load_dataset('openai/gsm8k', 'main', split='test')
        for i in rng.choice(len(gsm), CFG['n_gsm8k'], replace=False):
            corpus.append(gsm[int(i)]['question'])
        print(f'  GSM8K: {CFG["n_gsm8k"]} prompts')
    except Exception as e: print(f'  GSM8K skip: {e}')
    # SimpleQA
    try:
        sqa = load_dataset('basicv8vc/SimpleQA', split='test')
        for i in rng.choice(len(sqa), CFG['n_simpleqa'], replace=False):
            corpus.append(sqa[int(i)]['problem'])
        print(f'  SimpleQA: {CFG["n_simpleqa"]} prompts')
    except Exception as e: print(f'  SimpleQA skip: {e}')
    # Wikipedia (LONG, gives most residuals)
    try:
        wiki = load_dataset('wikimedia/wikipedia', '20231101.en', split='train', streaming=True)
        wiki_iter = iter(wiki)
        added = 0
        while added < CFG['n_wiki']:
            try:
                ex = next(wiki_iter)
                if len(ex['text']) > 1500:
                    corpus.append(ex['text'][:4500])
                    added += 1
            except StopIteration: break
        print(f'  Wikipedia: {added} long prompts')
    except Exception as e: print(f'  Wikipedia skip: {e}')
    print(f'\nTotal corpus: {len(corpus)} prompts')
    
    # Capture
    for prompt in tqdm(corpus, desc='capture'):
        captured.clear()
        inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=CFG['max_seq_len']).to(device)
        with torch.no_grad():
            _ = model(**inputs)
        for L in all_layers:
            if L in captured:
                r = captured[L].squeeze(0).float()
                all_residuals[L].append(r)
        torch.cuda.empty_cache()
    
    for L in all_layers:
        if all_residuals[L]:
            all_residuals[L] = torch.cat(all_residuals[L], dim=0)
            print(f'  L{L}: {all_residuals[L].shape}')
    
    torch.save(all_residuals, residuals_path)
    print(f'✓ Saved residuals to {residuals_path}')


## Phase C+ — Compute SAE features (resume-safe)


In [ ]:
features_path = OUT / 'features_per_layer.pt'
all_features = {}

# Validate features shape match residuals (handle stale cache)
stale_cache = False
if features_path.exists():
    cached = torch.load(features_path, weights_only=True)
    for L in all_layers:
        if L in cached and L in all_residuals:
            if cached[L].shape[0] != all_residuals[L].shape[0]:
                print(f'  ⚠️ L{L} features cached size {cached[L].shape[0]} != residuals {all_residuals[L].shape[0]} → recompute')
                stale_cache = True
                break
    if not stale_cache:
        print(f'✓ Loading cached features (validated)')
        all_features = cached
        for L in all_layers:
            if L in all_features:
                print(f'  L{L}: {all_features[L].shape}, sparsity {all_features[L].float().mean():.4f}')
    else:
        features_path.unlink()
if not features_path.exists() or stale_cache:
    for L in all_layers:
        if L not in SAE_ENCODERS or L not in all_residuals: continue
        W_enc = SAE_ENCODERS[L]['W_enc']
        b_enc = SAE_ENCODERS[L]['b_enc']
        R = all_residuals[L]
        n = R.shape[0]
        feature_topk = torch.zeros(n, CFG['d_sae'], dtype=torch.bool)
        batch = 512
        for i in tqdm(range(0, n, batch), desc=f'L{L}'):
            chunk = R[i:i+batch]
            logits = (chunk @ W_enc + b_enc).clamp(min=0)
            _, topk_idx = torch.topk(logits, k=CFG['k_sae'], dim=-1)
            feature_topk[i:i+batch].scatter_(-1, topk_idx, True)
        all_features[L] = feature_topk
        print(f'  L{L} features: {feature_topk.shape}, sparsity {feature_topk.float().mean():.4f}')
    torch.save(all_features, features_path)
    print(f'✓ Saved features to {features_path}')


## Phase D — Train probes (warm-start + ranking loss)


In [ ]:
class WarmStartProbe(nn.Module):
    """Probe initialized with SAE encoder. W shape: (d_model, d_sae)."""
    def __init__(self, W_enc, b_enc):
        super().__init__()
        self.W = nn.Parameter(W_enc.clone())
        self.b = nn.Parameter(b_enc.clone())
    def forward(self, x):
        return x @ self.W + self.b

def topk_ranking_loss(logits, target_active, k):
    pos_mask = target_active.bool()
    pos = (logits * pos_mask.float()).sum(-1) / k
    neg = (logits * (~pos_mask).float()).sum(-1) / (logits.shape[-1] - k)
    return F.softplus(-(pos - neg)).mean()

PAIRS = [(L_low, L_high) for (L_low, L_high) in [(11, 31), (11, 55)]
         if L_low in all_layers and L_high in all_layers]
print(f'Layer pairs to train: {PAIRS}')


In [ ]:
results = {}

for L_low, L_high in PAIRS:
    print(f'\n=== Training probe: L{L_low} → L{L_high} ===')
    R = all_residuals[L_low]
    F_target = all_features[L_high]
    n = R.shape[0]
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(CFG['random_seed']))
    n_train = int(0.8 * n)
    train_idx, test_idx = perm[:n_train], perm[n_train:]
    R_train, R_test = R[train_idx], R[test_idx]
    F_train, F_test = F_target[train_idx], F_target[test_idx]
    print(f'  train: {R_train.shape}, test: {R_test.shape}')
    
    probe = WarmStartProbe(SAE_ENCODERS[L_high]['W_enc'], SAE_ENCODERS[L_high]['b_enc']).to(device)
    optimizer = torch.optim.AdamW(probe.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    bs = CFG['batch_size_train']
    
    for epoch in range(CFG['n_epochs']):
        epoch_indices = torch.randperm(n_train)
        losses = []
        for i in range(0, n_train, bs):
            bi = epoch_indices[i:i+bs]
            x = R_train[bi].to(device)
            y = F_train[bi].float().to(device)
            logits = probe(x)
            loss = topk_ranking_loss(logits, y, CFG['k_sae'])
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            losses.append(loss.item())
        print(f'  Epoch {epoch+1}: loss {np.mean(losses):.4f}')
    
    # === Batched eval (no more 2GB matmul hang) ===
    probe.eval()
    bs_eval = CFG['batch_size_eval']
    n_test = R_test.shape[0]
    all_recall, all_recall_2k = [], []
    test_logits_chunks = []  # for AUROC if enabled
    
    print(f'  Eval over {n_test} samples in batches of {bs_eval}...')
    for i in tqdm(range(0, n_test, bs_eval), desc='eval'):
        x = R_test[i:i+bs_eval].to(device)
        with torch.no_grad():
            logits = probe(x).cpu()
        _, top_k_pred = torch.topk(logits, k=CFG['k_sae'], dim=-1)
        _, top_2k_pred = torch.topk(logits, k=2*CFG['k_sae'], dim=-1)
        pred_k = torch.zeros_like(logits, dtype=torch.bool)
        pred_k.scatter_(-1, top_k_pred, True)
        pred_2k = torch.zeros_like(logits, dtype=torch.bool)
        pred_2k.scatter_(-1, top_2k_pred, True)
        truth = F_test[i:i+bs_eval].bool()
        rec = (pred_k & truth).sum(-1).float() / CFG['k_sae']
        rec2 = (pred_2k & truth).sum(-1).float() / CFG['k_sae']
        all_recall.append(rec)
        all_recall_2k.append(rec2)
        if CFG['compute_per_feature_auroc']:
            test_logits_chunks.append(logits)
        del logits, pred_k, pred_2k
    
    recall = torch.cat(all_recall)
    recall_2k = torch.cat(all_recall_2k)
    print(f'  recall@k:  {recall.mean():.4f}')
    print(f'  recall@2k: {recall_2k.mean():.4f}')
    
    # Per-feature AUROC (optional, slow)
    auroc_mean = None
    if CFG['compute_per_feature_auroc']:
        from sklearn.metrics import roc_auc_score
        test_logits_full = torch.cat(test_logits_chunks).numpy()
        del test_logits_chunks
        F_test_np = F_test.numpy()
        rng_auroc = np.random.default_rng(CFG['random_seed'])
        feat_idx = rng_auroc.choice(CFG['d_sae'], CFG['auroc_n_features'], replace=False)
        aurocs = []
        for f in tqdm(feat_idx, desc='auroc'):
            if F_test_np[:, f].sum() < 5: continue
            try:
                aurocs.append(roc_auc_score(F_test_np[:, f], test_logits_full[:, f]))
            except: continue
        auroc_mean = float(np.mean(aurocs)) if aurocs else None
        print(f'  AUROC mean ({len(aurocs)} features): {auroc_mean:.4f}')
    
    results[(L_low, L_high)] = {
        'recall_at_k': float(recall.mean()),
        'recall_at_k_std': float(recall.std()),
        'recall_at_2k': float(recall_2k.mean()),
        'auroc_mean': auroc_mean,
        'n_test': len(recall),
    }
    
    # Save probe to Drive
    torch.save({'state_dict': probe.state_dict(),
                'L_low': L_low, 'L_high': L_high,
                'metrics': results[(L_low, L_high)]},
               OUT / f'probe_L{L_low}_to_L{L_high}.pt')
    del probe; torch.cuda.empty_cache(); gc.collect()


## Phase E — Verdict + viz + push


In [ ]:
import matplotlib.pyplot as plt

best_pair, best_metrics = max(results.items(), key=lambda kv: kv[1]['recall_at_k'])

if best_metrics['recall_at_k'] >= 0.85:
    verdict = '🟢 STRONG — PGAC viable, paper-grade result'
    paper_status = 'ship'
elif best_metrics['recall_at_k'] >= 0.5:
    verdict = '🟡 PARTIAL — top-2k strategy gives 2-3x speedup at quality cost'
    paper_status = 'ship_with_scope'
elif best_metrics['recall_at_2k'] >= 0.85:
    verdict = '🟡 ACCEPTABLE — top-2k preserves quality, ~2x speedup'
    paper_status = 'ship_partial'
else:
    verdict = '🔴 INSUFFICIENT — methodology revision needed'
    paper_status = 'rejected'

verdict_obj = {
    'experiment': 'PGAC Phase 2 v2 — probe training cross-layer',
    'corpus': {
        'gsm8k': CFG['n_gsm8k'], 'simpleqa': CFG['n_simpleqa'], 'wikipedia': CFG['n_wiki'],
        'total_residuals_per_layer': int(all_residuals[all_layers[0]].shape[0]),
    },
    'layer_pairs': [list(p) for p in results.keys()],
    'results_per_pair': {f'L{a}->L{b}': v for (a,b), v in results.items()},
    'best_pair': f'L{best_pair[0]}->L{best_pair[1]}',
    'verdict': verdict,
    'paper_status': paper_status,
}
(OUT / 'FINAL_VERDICT.json').write_text(json.dumps(verdict_obj, indent=2))
print(json.dumps(verdict_obj, indent=2))


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
labels = [f'L{a}→L{b}' for (a,b) in results.keys()]
rec_k = [results[p]['recall_at_k'] for p in results.keys()]
rec_2k = [results[p]['recall_at_2k'] for p in results.keys()]
x = np.arange(len(labels))
ax.bar(x - 0.2, rec_k, 0.4, label='recall@k', color='#10b981', alpha=0.85)
ax.bar(x + 0.2, rec_2k, 0.4, label='recall@2k (top-2k strategy)', color='#3b82f6', alpha=0.85)
ax.axhline(0.85, color='red', linestyle='--', alpha=0.5, label='Quality threshold')
ax.axhline(0.50, color='orange', linestyle=':', alpha=0.5, label='Min viable (top-2k)')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('Recall', fontsize=11)
ax.set_title('PGAC Phase 2 — probe top-k recall by layer pair', fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(OUT / 'pgac_phase2_results.png', dpi=170, bbox_inches='tight')
plt.show()


In [ ]:
from huggingface_hub import HfApi, create_repo
api = HfApi()
try: create_repo(CFG['output_repo'], repo_type='dataset', private=False, exist_ok=True, token=HF_TOKEN)
except Exception as e: print(e)

readme_lines = [
    '---',
    'license: apache-2.0',
    'tags:',
    '- pgac',
    '- probe',
    '- sae',
    '- qwen36-27b',
    '- adaptive-compute',
    '---',
    '',
    '# PGAC Phase 2 — Probe-Gated Adaptive Compute',
    '',
    f'**Verdict**: {verdict}',
    f'**Best pair**: {verdict_obj["best_pair"]}',
    '',
    '## Layer-pair results',',
    '',
    '| Pair | recall@k | recall@2k |',
    '|---|---|---|',
]
for (a, b), m in results.items():
    readme_lines.append(f'| L{a}→L{b} | {m["recall_at_k"]:.4f} | {m["recall_at_2k"]:.4f} |')
(OUT / 'README.md').write_text('\n'.join(readme_lines).replace(',\',\'',',\''))

try:
    api.upload_folder(folder_path=str(OUT), repo_id=CFG['output_repo'],
                      repo_type='dataset', token=HF_TOKEN,
                      commit_message=f'PGAC Phase 2: {verdict}',
                      allow_patterns=['README.md', 'FINAL_VERDICT.json',
                                      '*.png', 'probe_*.pt'])
    print('✓ pushed')
except Exception as e:
    print(f'HF push failed: {e}')


## Done — interpretation

- 🟢 STRONG: write up paper-grade result for NeurIPS MI Workshop Sep 2026
- 🟡 PARTIAL: ship top-2k strategy paper, ~2-3x speedup at modest quality cost
- 🔴 INSUFFICIENT: try MLP probe (more capacity), L31→L55 closer pair, or larger probe
